# Algorithmic Trading Strategy: walkthrough

A step-by-step tour of the engine: load data, form a signal, backtest it with an execution lag and trading costs, and test whether the result is distinguishable from luck.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from algo_trading import load_data, sma_crossover, xs_momentum, buy_and_hold, run_backtest, Costs, performance, SECTORS
from algo_trading import stats_tests as T
px, rf, info = load_data()
info['source'], info['start'], info['end']

('cached CSV', '1999-01-04', '2026-11-25')

## 1. Point-in-time universe
Sector funds that launched later have no price before launch, so they cannot be selected early. This avoids survivorship bias from building today's universe into the past.

In [2]:
pd.Series(info['first_valid'])

SPY     1999-01-04
XLB     1999-01-04
XLE     1999-01-04
XLF     1999-01-04
XLI     1999-01-04
XLK     1999-01-04
XLP     1999-01-04
XLU     1999-01-04
XLV     1999-01-04
XLY     1999-01-04
XLRE    2015-10-08
XLC     2018-06-19
dtype: str

## 2. A signal is a target weight decided at the close
The SMA 50/200 rule: hold SPY when the 50-day average is above the 200-day average, otherwise T-bills.

In [3]:
sig = sma_crossover(px['SPY'], 50, 200)
sig.dropna().tail()

,SPY
Date,
2026-11-19,1.0
2026-11-20,1.0
2026-11-23,1.0
2026-11-24,1.0
2026-11-25,1.0


## 3. Backtest with a one-day execution lag and 10 bps per trade
A signal formed at Monday's close trades at Tuesday's close; trading costs come out of equity on the trade date.

In [4]:
costs = Costs(commission_bps=5, slippage_bps=5)
sma = run_backtest(px, sig, rf, costs, lag=1)
spy = run_backtest(px, buy_and_hold(px[['SPY']]), rf, costs, lag=1)
start = sig.dropna().index[0]
pd.DataFrame({'SMA 50/200': performance(sma.returns.loc[start:], rf, spy.returns.loc[start:]),
              'SPY': performance(spy.returns.loc[start:], rf, spy.returns.loc[start:])}).round(3)

,SMA 50/200,SPY
CAGR,0.085,0.085
Volatility,0.135,0.192
Sharpe,0.525,0.422
Sortino,0.730,0.596
Max drawdown,-0.337,-0.552
Calmar,0.251,0.155
Longest drawdown (days),797.000,1656.000
Best year,0.323,0.323
Worst year,-0.082,-0.368
Daily hit rate,0.664,0.544


## 4. Look-ahead check
If we scramble every price after a cut-off date, nothing before the cut-off may change.

In [5]:
cut = px.index[3000]
shocked = px.copy(); shocked.loc[shocked.index > cut] *= 1.5
alt = run_backtest(shocked, sma_crossover(shocked['SPY'], 50, 200), rf, costs)
(alt.returns.loc[:cut] == sma.returns.loc[:cut]).all()

np.True_

## 5. Is the Sharpe improvement real?

In [6]:
r, b = sma.returns.loc[start:], spy.returns.loc[start:]
bs = T.bootstrap_sharpe_diff(r, b, rf, n_boot=500)
{'PSR': T.probabilistic_sharpe(r, rf), 'Sharpe diff': bs['observed'], 'CI': (bs['ci_low'], bs['ci_high']), 'P(<=0)': bs['p_le_zero']}

{'PSR': 0.996584687944799,
 'Sharpe diff': 0.1032237266552054,
 'CI': (-0.18989168747084395, 0.41459858185305737),
 'P(<=0)': 0.236}

## 6. Sector momentum
Hold the 3 sectors with the best 12-1 month return, rebalanced monthly; a sector with negative momentum is replaced by cash.

In [7]:
xs = run_backtest(px, xs_momentum(px[[s for s in SECTORS if s in px]]), rf, costs)
pd.Series(performance(xs.returns.loc[start:], rf, b)).round(3)

CAGR                          0.079
Volatility                    0.171
Sharpe                        0.416
Sortino                       0.578
Max drawdown                 -0.348
Calmar                        0.226
Longest drawdown (days)    1106.000
Best year                     0.370
Worst year                   -0.216
Daily hit rate                0.562
Beta                          0.689
Alpha / yr                    0.015
dtype: float64